# PLE Line Repump Notebook (DLC Pro Active Polling)

Controls the **iBeam Smart** laser as a repump source during PLE scans.

### Features
- CW repump on/off + power control
- **Line repump**: polls the Toptica DLC Pro via the `toptica.lasersdk` to detect the end of a scan line, applies a repump pulse, and manually triggers the next scan line.

### Prerequisites
- qudi running with `ibeam_smart`, `dl_pro`, `ple_gui`, `laser_scanner_logic`
- `toptica.lasersdk` installed in the python environment

## 1 · Imports & qudi module handles

In [1]:
import time
import threading
from PySide2 import QtCore
from toptica.lasersdk.dlcpro.v2_0_3 import DLCpro, NetworkConnection

# ── qudi modules available in the Jupyter kernel namespace ────────────────
# (these are injected by qudi's namespace server; adjust names if needed)
hw   = ibeam_smart          # iBeamSmart hardware object
scan = laser_scanner_logic  # PLEScannerLogic
gui  = ple_gui              # PLEScanGui
dlc_hw = dl_pro             # needed to get the IP address (tcp_address)

print("iBeam power:", hw.getPower(), "µW")
print("DLC Pro IP:", dlc_hw.tcp_address)

iBeam power: 100000.0 µW
DLC Pro IP: 129.69.46.10


## 2 · CW repump helpers

In [2]:
# ── CW power & toggle ────────────────────────────────────────────────────

def repump_set_power(power_uW: float):
    """Set iBeam power in µW."""
    hw.setPower(power_uW)
    print(f"iBeam power set to {power_uW} µW")

def repump_on():
    """Turn iBeam CW on."""
    hw.enable()
    print("iBeam ON")

def repump_off():
    """Turn iBeam CW off."""
    hw.disable()
    print("iBeam OFF")

def repump_pulse(duration_s: float = 0.5):
    """Fire a single repump pulse of given duration."""
    hw.enable()
    time.sleep(duration_s)
    hw.disable()
    print(f"Repump pulse: {duration_s:.3f} s")

## 3 · Custom Line Repump scan loop

Instead of hooking into `sigRepeatScan`, we take control of the DLC pro directly:
1. Disable DLC pro continuous mode
2. Start PLE scan in qudi (which will wait since continuous mode is off)
3. Manually trigger each scan line, poll for completion, and repump

In [3]:
# ── Configuration ────────────────────────────────────────────────────────
LINE_REPUMP_POWER_uW = 5000.0
LINE_REPUMP_DURATION = 0.5
CW_POWER_uW          = 0.0     # Power strictly between lines/scans

_stop_scan_flag = False

def do_custom_ple_scan(lines: int = 10, scan_range=None, with_line_repump: bool = True, conditional_repump: bool = False):
    """
    If conditional_repump is True, performs a fit using the GUI selected model.
    If the fit succeeds, it skips the repump. If it fails, repump is applied.
    """
    """
    Run a PLE scan where we explicitly control the DLC pro wide_scan to insert 
    repump pulses between scan lines.
    """
    global _stop_scan_flag
    _stop_scan_flag = False

    if scan_range is not None:
        gui.sigScanSettingsChanged.emit(
            {'range': {gui.scan_axis: tuple(int(v) for v in scan_range)}}
        )
        time.sleep(0.2)

    # Set repeats in GUI and start qudi scan
    # (The scan won't actually step through lines because we're about to disable continuous_mode)
    gui._mw.number_of_repeats_SpinBox.setValue(lines)
    gui._mw.number_of_repeats_SpinBox.editingFinished.emit()
    time.sleep(0.2)
    gui._mw.actionToggle_scan.setChecked(True)
    gui.toggle_scan()
    time.sleep(0.5)  # give qudi time to enter the locked scanning state

    # Connect to DLC Pro and take manual control of the scan lines
    try:
        with DLCpro(NetworkConnection(dlc_hw.tcp_address)) as dlc:
            # Setup DLC pro for manual per-line triggering
            dlc.laser1.wide_scan.trigger.output_enabled.set(True)
            dlc.laser1.wide_scan.continuous_mode.set(False)
            
            print(f"Starting sequential line scan ({lines} lines)...")
            
            # Ensure baseline CW state
            if CW_POWER_uW > 0:
                hw.setPower(CW_POWER_uW)
                hw.enable()
            else:
                hw.disable()
                
            for current_line in range(lines):
                        
                # If user called stop_scan(), break early
                if _stop_scan_flag or scan.module_state() != 'locked':
                    print("Scan interrupted.")
                    break
                    
                # 1. Start the scan line on the DLC pro
                laser_state = dlc.laser1.wide_scan.state.get()
                if laser_state == 0:
                    dlc.laser1.wide_scan.start()
                    
                time.sleep(0.5)
                
                # 2. Wait for the line to complete on the hardware
                laser_state = dlc.laser1.wide_scan.state.get()
                while laser_state != 0:
                    if _stop_scan_flag:
                        break
                    time.sleep(1)
                    laser_state = dlc.laser1.wide_scan.state.get()
                    
                time.sleep(1) # Extra buffer after line finishes
                
                # 3. Perform Repump
                do_repump_this_line = with_line_repump
                
                if with_line_repump and conditional_repump and not _stop_scan_flag:
                    channel = scan._channel
                    fit_config = gui._fit_dockwidget.fit_widget.selection_combobox.currentText()
                    
                    if fit_config != "No Fit":
                        # Perform the fit on the latest line data
                        scan.do_fit(fit_config, channel, averaged=False)
                        fit_res = scan.fit_results.get(channel)
                        
                        if fit_res is None:
                            print(f"  Line {current_line + 1}: Fit \"{fit_config}\" FAILED. Defect likely ionized. Repumping.")
                            do_repump_this_line = True
                        else:
                            print(f"  Line {current_line + 1}: Fit \"{fit_config}\" succeeded. Skipping repump.")
                            do_repump_this_line = False
                    else:
                        print(f"  Line {current_line + 1}: \"No Fit\" selected. Cannot do conditional repump. Defaulting to YES.")
                        do_repump_this_line = True
                        
                if do_repump_this_line and not _stop_scan_flag:
                    print(f"  Line {current_line + 1}/{lines} done. Applying {LINE_REPUMP_DURATION}s repump.")
                    try:
                        hw.setPower(LINE_REPUMP_POWER_uW)
                        hw.enable()
                        time.sleep(LINE_REPUMP_DURATION)
                    finally:
                        if CW_POWER_uW > 0:
                            hw.setPower(CW_POWER_uW)
                            hw.enable()
                        else:
                            hw.disable()
                else:
                    print(f"  Line {current_line + 1}/{lines} done.")
                    
    except Exception as e:
        print(f"Error communicating with DLC Pro: {e}")
    finally:
        # Clean up
        hw.disable()
        gui._mw.actionToggle_scan.setChecked(False)
        if scan.module_state() == 'locked':
            gui.toggle_scan()
        print("Scan loop concluded.")

def stop_scan():
    """Abort the running custom scan loop safely."""
    global _stop_scan_flag
    _stop_scan_flag = True
    print("Stopping scan...")

## 4 · Quick controls

In [4]:
# ── Set power and test a single pulse ────────────────────────────────────
repump_set_power(LINE_REPUMP_POWER_uW)
repump_pulse(duration_s=0.2)

iBeam power set to 5000.0 µW
Repump pulse: 0.200 s


In [7]:
# ── Run Custom PLE scan WITH line repump ─────────────────────────────────
LINE_REPUMP_POWER_uW = 1000.0   # µW
LINE_REPUMP_DURATION = 0.25      # s
CW_POWER_uW          = 0.0      # power while scanning

do_custom_ple_scan(lines=200, with_line_repump=True, conditional_repump=True)

Starting sequential line scan (200 lines)...
  Line 1/200 done. Applying 0.25s repump.
  Line 2/200 done. Applying 0.25s repump.
  Line 3/200 done. Applying 0.25s repump.
  Line 4/200 done. Applying 0.25s repump.
  Line 5/200 done. Applying 0.25s repump.
  Line 6/200 done. Applying 0.25s repump.
  Line 7/200 done. Applying 0.25s repump.
  Line 8/200 done. Applying 0.25s repump.
  Line 9/200 done. Applying 0.25s repump.
  Line 10/200 done. Applying 0.25s repump.
  Line 11/200 done. Applying 0.25s repump.
  Line 12/200 done. Applying 0.25s repump.
  Line 13/200 done. Applying 0.25s repump.
  Line 14/200 done. Applying 0.25s repump.
  Line 15/200 done. Applying 0.25s repump.
  Line 16/200 done. Applying 0.25s repump.
  Line 17/200 done. Applying 0.25s repump.
  Line 18/200 done. Applying 0.25s repump.
  Line 19/200 done. Applying 0.25s repump.
  Line 20/200 done. Applying 0.25s repump.
  Line 21/200 done. Applying 0.25s repump.
  Line 22/200 done. Applying 0.25s repump.
  Line 23/200 done

In [ ]:
# ── Run Custom PLE scan WITHOUT line repump ──────────────────────────────
do_custom_ple_scan(lines=5, with_line_repump=False)

In [ ]:
# ── Emergency stop ───────────────────────────────────────────────────────
stop_scan()

In [ ]:
# ── Turn off iBeam manually ──────────────────────────────────────────────
repump_off()